# 01 — A question that splits the data: impurity

*Notebook 1 of 5 — Decision Trees*

Welcome to decision trees. Logistic regression drew one straight line; a tree does something
different and wonderfully human — it asks a sequence of yes/no questions. In this notebook we build
the very first move a tree makes: a single split, and the measure it uses to choose one. You will do
it entirely by hand before meeting the library.

**Prerequisites**

- Module 00 — features and the feature space (NB 02), the train/test split (NB 04), accuracy (NB 06).
- Chapter 03 — Logistic Regression (NB 6): a straight-line boundary *underfits* when the truth is
  curved. That limitation is the door to trees.

**What you'll be able to do**

- Measure how class-mixed a group of points is, with the Gini index and with entropy.
- Score a yes/no split by how much it *reduces* that impurity.
- Find the best split on a feature by scanning its thresholds.
- Recognise that this single move is what `DecisionTreeClassifier` makes first.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.tree import DecisionTreeClassifier  # used only for the parity checks

from ml_course import datasets, viz
from ml_course.colors import CLASS_CYCLE, COLORS

np.random.seed(0)
viz.use_course_style()

# The binary penguin subset, in raw millimetres (a tree is scale-invariant).
df = datasets.load_penguins()
X, y = datasets.penguins_xy(df)

print(f"{len(df)} penguins; features = {list(X.columns)}")
print(y.value_counts())

## Where we are

The course has, so far, mostly drawn **lines**. Nearest centroid placed a straight boundary halfway
between two class means; logistic regression fitted a weighted line and read a probability off it.
Both carve the feature space with a single global cut.

A decision tree carves it differently: it asks a **yes/no question** about one feature — "is the
flipper shorter than 206 mm?" — and splits the data into the points that answer yes and the points
that answer no. Today we build that one move, and the number a tree uses to decide *which* question
to ask.

We work on the two-species penguin subset (Adélie vs Gentoo) with two measurements, in their **raw
units** — millimetres. We keep them raw on purpose: a split is a threshold, so the scale of a feature
does not matter to a tree. We come back to that at the end. Nothing is fitted with a library until
the final parity check; the mechanism comes first.

## What makes a split good?

Picture a group of points — a **node**. If it holds both species in equal measure, it is **mixed**;
if it holds almost only one species, it is **pure**. A good question takes a mixed node and sends each
answer into a purer group.

To *choose* between questions we need to turn "how mixed" into a number. That number is called
**impurity**.

In [ ]:
fig = viz.plot_feature_histograms(
    df,
    ["bill_length_mm", "flipper_length_mm"],
    by="species",
)
plt.show()

### Read the figure

Each panel shows, for one measurement, how the two species are distributed. Both measurements
separate Adélie from Gentoo to a degree — the coloured humps sit in different places — but
`flipper_length_mm` separates them more cleanly: its two humps barely overlap, while the
`bill_length_mm` humps share a wider middle band. That is a hint that a single cut on flipper length
will do better. We will not trust the hint — we will measure it.

## Impurity: two ways to measure "mixed"

Let $p$ be the proportion of one class in a node (so $1 - p$ is the other). Two classic impurity
measures:

- **Gini index**: $G = 1 - \sum_k p_k^2$. Read it as the chance of mislabelling a point if you
  guessed classes at random in the node's own proportions — lowest when one class dominates.
- **Entropy**: $H = -\sum_k p_k \log_2 p_k$, in **bits** — the average "surprise" of a point's label.

Both are **0** when a node is pure (no surprise, no mislabels) and **largest** when the classes sit
at 50/50. Let us plot them.

In [ ]:
def gini(labels):
    """Gini impurity 1 - sum_k p_k^2 of a set of binary labels (0/1)."""
    labels = np.asarray(labels)
    if labels.size == 0:
        return 0.0
    p = labels.mean()
    return 1.0 - p**2 - (1.0 - p) ** 2


def entropy(labels):
    """Shannon entropy -sum_k p_k log2 p_k (in bits) of binary labels (0/1)."""
    labels = np.asarray(labels)
    if labels.size == 0:
        return 0.0
    p = labels.mean()
    if p == 0.0 or p == 1.0:
        return 0.0
    return -(p * np.log2(p) + (1.0 - p) * np.log2(1.0 - p))


# The two measures as a function of the proportion p of one class.
p_grid = np.linspace(0.0, 1.0, 201)
gini_curve = 1.0 - p_grid**2 - (1.0 - p_grid) ** 2
with np.errstate(divide="ignore", invalid="ignore"):
    ent_raw = -(p_grid * np.log2(p_grid) + (1.0 - p_grid) * np.log2(1.0 - p_grid))
entropy_curve = np.nan_to_num(ent_raw)  # entropy -> 0 at the pure ends p = 0 and p = 1

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(p_grid, gini_curve, color=COLORS["class_a"], linewidth=2, label="Gini")
ax.plot(p_grid, entropy_curve, color=COLORS["class_b"], linewidth=2, label="entropy (bits)")
ax.axvline(0.5, color=COLORS["muted"], linestyle="--", linewidth=1)
ax.set_xlabel("proportion p of one class")
ax.set_ylabel("impurity")
ax.legend()
plt.show()

### Read the figure

Both curves rise from 0 at a pure node ($p = 0$ or $p = 1$), climb to a single peak at $p = \tfrac12$,
and fall back. The Gini index tops out at **0.5**; entropy tops out at **1 bit**. They have the same
shape and tell the same story — "how far is this node from pure" — which is why a tree can use either.
We will use Gini (the common default) and check that entropy agrees.

In [ ]:
# Encode the label as 1 = Gentoo, 0 = Adelie, then measure the root node's impurity.
y_bin = (y == "Gentoo").astype(int).to_numpy()

gini_root = gini(y_bin)
entropy_root = entropy(y_bin)
print(f"root Gini    = {gini_root:.4f}")
print(f"root entropy = {entropy_root:.4f} bits")

# The library computes the very same root impurity.
stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X, y_bin)
print(f"sklearn root impurity (Gini) = {stump.tree_.impurity[0]:.4f}")

### Read the result

Our by-hand Gini and the value `DecisionTreeClassifier` stores for the root agree to four decimals —
**0.4948**. The root is close to the 0.5 ceiling: with 151 Adélie and 123 Gentoo, it is almost as
mixed as a two-class node can be. Entropy says the same in its own units, **0.9925 bits**, a hair below
the 1-bit maximum. A good first question should drop this number sharply.

## The best split is the biggest drop in impurity

A threshold split on a feature sends every point with `feature ≤ t` to a **left** child and the rest
to a **right** child. How good is it? Compare the impurity *before* and *after*:

$$\text{decrease} = G(\text{parent}) - \frac{n_L}{n}\,G(\text{left}) - \frac{n_R}{n}\,G(\text{right})$$

We subtract the **sample-weighted** average of the children's impurity (a child with more points
counts more). For entropy this quantity has a name — **information gain**. The best split on a feature
is the threshold that makes this decrease as large as possible.

In [ ]:
def impurity_decrease(values, labels, threshold, impurity=gini):
    """Impurity decrease of 'values <= threshold': parent minus weighted children."""
    values = np.asarray(values)
    labels = np.asarray(labels)
    left = labels[values <= threshold]
    right = labels[values > threshold]
    n = labels.size
    weighted = (left.size * impurity(left) + right.size * impurity(right)) / n
    return impurity(labels) - weighted


def scan_thresholds(values, labels, impurity=gini):
    """Score every midpoint between sorted unique values; return (thresholds, decreases)."""
    uniq = np.sort(np.unique(values))
    thresholds = (uniq[:-1] + uniq[1:]) / 2.0
    decreases = np.array(
        [impurity_decrease(values, labels, t, impurity) for t in thresholds]
    )
    return thresholds, decreases


flip = X["flipper_length_mm"].to_numpy()
bill = X["bill_length_mm"].to_numpy()
flip_t, flip_d = scan_thresholds(flip, y_bin)
bill_t, bill_d = scan_thresholds(bill, y_bin)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.0), sharey=True)
for ax, name, t, d in [
    (ax1, "flipper_length_mm", flip_t, flip_d),
    (ax2, "bill_length_mm", bill_t, bill_d),
]:
    ax.plot(t, d, color=COLORS["model"], linewidth=2)
    ax.scatter([t[d.argmax()]], [d.max()], color=COLORS["highlight"], zorder=5, s=50)
    ax.set_title(f"{name}: peak at {t[d.argmax()]:.1f}")
    ax.set_xlabel("split threshold")
ax1.set_ylabel("Gini impurity decrease")
plt.show()

### Read the figure

For each feature we slid the threshold across every gap between neighbouring values and measured the
impurity decrease. Each curve rises to a single broad peak (bill length's is a little ragged at the
top, where neighbouring thresholds score almost the same): cut too low or too high and the split
barely helps; cut near the peak and impurity falls a lot. The **peak** of each hump is that feature's
best split. Flipper length's peak (around 206 mm) sits higher than bill length's (around 43 mm) — the
measurement that separated more cleanly also makes the better cut.

In [ ]:
flip_best_t = flip_t[flip_d.argmax()]
bill_best_t = bill_t[bill_d.argmax()]
print(f"flipper_length_mm: best split <= {flip_best_t:.2f}  (Gini decrease {flip_d.max():.4f})")
print(f"bill_length_mm:    best split <= {bill_best_t:.2f}  (Gini decrease {bill_d.max():.4f})")

# Entropy (information gain), scanned the same way, prefers the same thresholds.
flip_te, flip_de = scan_thresholds(flip, y_bin, impurity=entropy)
bill_te, bill_de = scan_thresholds(bill, y_bin, impurity=entropy)
print(
    f"entropy info-gain best: flipper <= {flip_te[flip_de.argmax()]:.2f}, "
    f"bill <= {bill_te[bill_de.argmax()]:.2f}"
)

### Read the result

The numbers confirm the picture: splitting on **`flipper_length_mm` ≤ 206** drops Gini by **0.4732**,
more than the best bill-length split (**0.4044**). And entropy, scanned the same way, points to the
*same two thresholds* — the choice of impurity measure does not change the best question here.

Notice what happened: we did not *choose* flipper length by eye. We let the impurity criterion
**measure** which feature makes the better single cut, and it named flipper. Back in Chapter 03 we
introduced the sigmoid on `bill_length` as a narrative choice; here the scored data picks the feature
for us. Letting the criterion choose the split is the idea a whole tree is built on.

In [ ]:
threshold = flip_best_t  # 206.0 mm
left = y_bin[flip <= threshold]
right = y_bin[flip > threshold]


def mix(labels):
    n = labels.size
    gentoo = int(labels.sum())
    return n, n - gentoo, gentoo, gini(labels)  # n, Adelie, Gentoo, Gini


nL, aL, gL, giniL = mix(left)
nR, aR, gR, giniR = mix(right)
weighted = (nL * giniL + nR * giniR) / y_bin.size
print(f"left  (flipper <= {threshold:.0f}): {aL} Adelie / {gL} Gentoo, Gini {giniL:.4f}")
print(f"right (flipper >  {threshold:.0f}): {aR} Adelie / {gR} Gentoo, Gini {giniR:.4f}")
print(f"weighted children Gini = {weighted:.4f}  ->  decrease = {gini_root - weighted:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
for i, sp in enumerate(["Adelie", "Gentoo"]):
    m = (y == sp).to_numpy()
    ax1.scatter(
        X.loc[m, "bill_length_mm"],
        X.loc[m, "flipper_length_mm"],
        color=CLASS_CYCLE[i],
        edgecolor=COLORS["text"],
        linewidth=0.5,
        s=40,
        label=sp,
    )
ax1.axhline(
    threshold,
    color=COLORS["highlight"],
    linestyle="--",
    linewidth=2,
    label=f"flipper = {threshold:.0f}",
)
ax1.set_xlabel("bill_length_mm")
ax1.set_ylabel("flipper_length_mm")
ax1.legend()

names = [f"left\n(<= {threshold:.0f})", f"right\n(> {threshold:.0f})"]
xpos = np.arange(2)
ax2.bar(
    xpos,
    [aL, aR],
    color=CLASS_CYCLE[0],
    edgecolor=COLORS["text"],
    linewidth=0.5,
    label="Adelie",
)
ax2.bar(
    xpos,
    [gL, gR],
    bottom=[aL, aR],
    color=CLASS_CYCLE[1],
    edgecolor=COLORS["text"],
    linewidth=0.5,
    label="Gentoo",
)
ax2.set_xticks(xpos, names)
ax2.set_ylabel("count")
ax2.legend()
plt.show()

### Read the figure

On the left, the dashed line is our chosen split: a single horizontal cut at flipper = 206 mm. Almost
every Adélie falls below it and almost every Gentoo above. On the right, the stacked bars make the
children's purity exact: the left child is 149 Adélie to 1 Gentoo, the right is 122 Gentoo to 2
Adélie. Each child's Gini is tiny (about 0.01 and 0.03), and their weighted average — 0.0216 — is what
we subtract from the parent's 0.4948 to reach the 0.4732 decrease. That near-purity *is* the large
drop, drawn.

In [ ]:
# Parity: the library's depth-1 tree makes exactly the split we found by hand.
stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X, y_bin)
root_feature = X.columns[stump.tree_.feature[0]]
root_threshold = stump.tree_.threshold[0]
print(f"sklearn stump root: {root_feature} <= {root_threshold:.2f}")
print(f"root impurity     : {stump.tree_.impurity[0]:.4f}")
print(f"training accuracy : {stump.score(X, y_bin):.4f}")

# Scale-invariance: standardizing the feature only rescales the threshold.
flip_std = (flip - flip.mean()) / flip.std()
raw_best = scan_thresholds(flip, y_bin)[1].max()
std_best = scan_thresholds(flip_std, y_bin)[1].max()
print(f"best decrease on flipper -- raw: {raw_best:.4f}, standardized: {std_best:.4f}")

### Read the result

The library's first move is the split we found by hand: root question `flipper_length_mm ≤ 206`, root
impurity `0.4948`, training accuracy `0.9891` — one well-chosen question already labels all but three
penguins correctly. And notice the last line: the best impurity decrease is **identical** on the raw
feature and on the standardized one. A split only asks "which side of a threshold?", so rescaling the
feature merely rescales the threshold — the partition, and the whole tree, is unchanged. That is why
we never standardized here, and why the scale trap that bit k-NN (Chapter 01) does not touch a tree.

## Your turn

1. **Compute a Gini by hand.** A node holds 30 Adélie and 10 Gentoo. What is its Gini index? (Use
   $G = 1 - p_A^2 - p_G^2$.) Is it more or less pure than the root?
2. **Scan the other feature.** Call `scan_thresholds(bill, y_bin)` and confirm its best split is at
   ≈ 43.25. Then take the *left* child of the flipper split and scan a feature inside it — what is the
   best second question for the penguins that answered "flipper ≤ 206"?
3. **Why those extremes?** Argue from the formula why a **pure** node has Gini 0, why a **50/50** node
   is the most impure a two-class node can be, and why entropy's maximum there is exactly **1 bit**.

## What you built

- **Impurity** — a number for how class-mixed a node is, with two measures: the **Gini index** and
  **entropy**. Both are 0 at a pure node and largest at 50/50.
- The **impurity decrease** that scores a split: the parent's impurity minus the sample-weighted
  impurity of its two children (this is **information gain** for entropy).
- A way to find the **best split** on a feature — scan every threshold, keep the largest decrease —
  and a comparison across features that crowned `flipper_length_mm ≤ 206`.
- The reassurance that this single move is what `DecisionTreeClassifier` makes first, and that a
  split is **scale-invariant**.

**Vocabulary:** node · impurity · Gini index · entropy · split · threshold · child node · impurity
decrease / information gain · stump · scale-invariance.

## Going further (optional)

- **Gini or entropy?** They rarely disagree on the best split (they did not here). Gini is the common
  default mostly because it is cheaper to compute — no logarithm.
- **Binary splits.** The CART algorithm we are following always splits a node in two; some older
  algorithms allowed multiway splits.
- **Greedy, not optimal.** We picked the single best split. Building the *globally* best tree is
  NP-hard, so trees are grown one greedy split at a time — which is what the next notebook does.

## References

- Breiman, Friedman, Olshen & Stone (1984). *Classification and Regression Trees (CART).* Wadsworth.
- Quinlan, J. R. (1986). *Induction of decision trees.* Machine Learning 1, 81–106.
  DOI: 10.1007/BF00116251.
- Hastie, Tibshirani & Friedman (2009). *The Elements of Statistical Learning*, §9.2.
  DOI: 10.1007/978-0-387-84858-7.
- James, Witten, Hastie & Tibshirani (2021). *An Introduction to Statistical Learning*, §8.1.
  DOI: 10.1007/978-1-0716-1418-1.

*Previous: Module 03 — Logistic Regression (NB 6) · Next: 02 — Growing a tree, and reading it.*